In [ ]:
import numpy as np
from itertools import zip_longest
def fromarrays(a, name):
    if len(a)<2:
        return a[0].view([(name[0],a[0].dtype)])
    i = min(next(i for i,(a,b)in enumerate(zip_longest(a[0].shape,b.shape+(0,)))if a!=b)for b in a[1:])
    b = np.empty(a[0].shape[:i],[(name,a.dtype,a.shape[i:])for name,a in zip(name,a)])
    for name,a in zip(name,a):
        b[name] = a
    return b
def unpackbits(a):
    if a.dtype.base=='B':
        return np.unpackbits(a,-1).view('?')
    if a.dtype.names and any((_:=list(zip(*((_:=unpackbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def packbits(a):
    if a.dtype.names and a.dtype[0]=='?':
        return np.packbits(a[...,None].view('?'),-1)
    if a.dtype.names and any((_:=list(zip(*((_:=packbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def tobj(a):
    return a.tolist()if not a.dtype.names else[tobj(a)for a in a]if a.ndim else{i:tobj(a[i])for i in a.dtype.names}
def fromobj(a, type, d=0):
    return [fromobj(a,type,d)for a in a]if isinstance(a,list)else tuple(fromobj(a[i],type,d)if i in a else d for i,(type,_)in type.fields.items())if isinstance(a,dict)else a

In [ ]:
from fastapi import FastAPI, Request, Response
from json import loads
api = FastAPI()
@api.post('/tobytes')
def tobytes(a: dict, type: str) -> bytes:
    type = loads(type)
    return packbits(np.array(fromobj(a, np.dtype(type)), type)).tobytes()
@api.post('/frombytes')
def frombytes(a:bytes, type: str, compactype:str|None=None) -> dict:
    type = loads(type)
    return toobj(unpackbits(np.frombuffer(a, loads(compactype))).view(type) if compactype else np.frombuffer(a, type))

In [ ]:
np.dtype([['a','B']])

In [ ]:
from json import loads
np.dtype(loads('[["a","B"]]'))

In [ ]:
np.dtype([('a','B')])

In [ ]:
def tobytes(a: dict, type: DTypeLike) -> bytes: ...
def frombytes(a: bytes, type: DTypeLike, compacttype: DTypeLike) -> dict: ...

How to use these with fastapi? type and compacttype can be very long, so they should probably be 


In [ ]:
d = np.dtype('2B,f,2f')
D = np.dtype([
    ('flags',[
        ('a','?'),
        ('b','?'),
        ('c','?'),
        ('d','?'),
        ('e','?'),
        ('f','?'),
        ('g','?'),
        ('h','?'),
        ('i','?'),
        ('j','?'),
        ('k','?'),
        ('l','?'),
        ('m','?'),
        ('n','?'),
        ('o','?'),
        ('p','?')
    ]),
    ('val1','f'),
    ('val2','2f')
])
raw = bytearray(56)
a = unpackbits(np.frombuffer(raw,d)).view(D)

In [ ]:
np.dtype({'f0': (np.dtype('bool'), (16,)), 'f1': (np.dtype('float32'), ()), 'f2': (np.dtype('float32'), (2,))})

In [ ]:
import numpy as np
from itertools import zip_longest
def fromarrays(a, name):
    if len(a)<2:
        return a[0].view([(name[0],a[0].dtype)])
    i = min(next(i for i,(a,b)in enumerate(zip_longest(a[0].shape,b.shape+(0,)))if a!=b)for b in a[1:])
    b = np.empty(a[0].shape[:i],[(name,a.dtype,a.shape[i:])for name,a in zip(name,a)])
    for name,a in zip(name,a):
        b[name] = a
    return b
def unpackbits(a):
    if a.dtype.base=='B':
        return np.unpackbits(a,-1).view('?')
    if a.dtype.names and any((_:=list(zip(*((_:=unpackbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def tobj(a):
    return a.tolist()if not a.dtype.names else[tobj(a)for a in a]if a.ndim else{i:tobj(a[i])for i in a.dtype.names}
def fromobj(a, type, d=0):
    return [fromobj(a,type,d)for a in a]if isinstance(a,list)else tuple(fromobj(a[i],type,d)if i in a else d for i,(type,_)in type.fields.items())if isinstance(a,dict)else a
def packbits(a):
    if a.dtype.names and a.dtype[0]=='?':
        return np.packbits(a[...,None].view('?'),-1)
    if a.dtype.names and any((_:=list(zip(*((_:=packbits(a[i]),_ is not a[i])for i in a.dtype.names))))[1]):
        return fromarrays(_[0], a.dtype.names)
    return a
def unjsonifysparse(a, type):
    if isinstance(a, list):
        stype = np.dtype((type.base,type.shape[1:]))
        size = type.itemsize//(type.shape or[1])[0]
        b,name,stype,of = zip(*[(a,str(i),a.dtype,i*size)for i,a in enumerate(a)if a is not None for a in[unjsonifysparse(a,stype)]])
        return np.array(b,None if all(a is not None for a in a)and all(i==stype[0]for i in stype[1:])else{'names':name,'formats':stype,'offsets':of}|({'itemsize':type.itemsize}if type else{}))
    if isinstance(a, dict):
        a,name,stype,of = zip(*[(b,i,(b.dtype,b.shape),of)for i,(type,of)in type.fields.items()for b in[a.get(i)]if b is not None for b in[unjsonifysparse(b,type)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    return type.type(a)
def tobytesparse(a):
    t = ~a[None].view('B')
    t.view(a.dtype)[:] = a
    a = a[None].view('B')
    t = (t==a).nonzero()[0]
    i = np.concat([[0],(t[1:]-t[:-1]>1).nonzero()[0]+1,[len(t)]])
    return [(t,a[t:t+i])for t,i in zip(t[i[:-1]],i[1:]-i[:-1])]
def slicesparse(a, i):
    a = a[::-1]
    for i,j in i:
        o = []
        while a:
            if len(_:=a[-1][1][max(i-a[-1][0],0):max(j-a[-1][0],0)]):
                o.append((max(a[-1][0]-i,0),_))
            if j-a[-1][0]<len(a[-1][1]):
                break
            a.pop()
        yield o
def frombytesparse(a, type):
    if type.shape:
        stype = np.dtype((type.base,type.shape[1:]))
        size = type.itemsize//type.shape[0]
        a,name,stype,of = zip(*[(a,str(i),(a.dtype,a.shape),i*size)for i,a in enumerate(slicesparse(a,zip(range(0,type.shape[0]*size,size),range(size,(type.shape[0]+1)*size,size))))if a for a in[frombytesparse(a,stype)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    if type.names:
        a,name,stype,of = zip(*[(a,i,(a.dtype,a.shape),of)for (i,(type,of)),a in zip(type.fields.items(),slicesparse(a,[(of,of+type.itemsize)for _,(type,of)in type.fields.items()]))if a for a in[frombytesparse(a,type)]])
        return np.array(a, {'names':name,'formats':stype,'offsets':of,'itemsize':type.itemsize})
    return np.frombuffer(a[0][1], type)[0]
def bytemask(a, t='B'):
    return a.filled((~a.data.view(t)).view(a.dtype)).view(t)==a.data.view(t)
def putmask(a, b, t='B'):
    return np.putmask(b.view(t), bytemask(a), a.view(t))
d = np.dtype('2B,f,2f')
D = np.dtype([
    ('flags',[
        ('a','?'),
        ('b','?'),
        ('c','?'),
        ('d','?'),
        ('e','?'),
        ('f','?'),
        ('g','?'),
        ('h','?'),
        ('i','?'),
        ('j','?'),
        ('k','?'),
        ('l','?'),
        ('m','?'),
        ('n','?'),
        ('o','?'),
        ('p','?')
    ]),
    ('val1','f'),
    ('val2','2f')
])
raw = bytearray(56)
a = unpackbits(np.frombuffer(raw,d)).view(D)

In [ ]:
np.frombuffer(raw,d)